# P87 — Ensayo para resolver un problema en la doctrina de las probabilidades

## 1. Título y paper

**Paper:** *An Essay towards solving a Problem in the Doctrine of Chances*  
**Autoría:** Thomas Bayes, Richard Price (editor)  
**Año y venue:** 1763 · Philosophical Transactions of the Royal Society, 53, 370–418  
**Nivel:** L2 · **Motor:** `bayes`  
**Ficha completa:** [`P87_bayes`](../../papers/foundational/P87_bayes/README.md)

**Hito:** La regla que invierte el condicional: pasar de «qué esperaría ver si la hipótesis fuese cierta» a «cuán probable es la hipótesis dado lo que he visto».

- [doi:10.1098/rstl.1763.0053](https://doi.org/10.1098/rstl.1763.0053)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La probabilidad sabía calcular qué datos esperar dada una causa conocida. La pregunta inversa —qué causa es probable dados los datos observados— no tenía tratamiento, y es la que hace falta para aprender de la experiencia.
2. Ejecutar una implementación mínima de la propuesta: Tratar la causa desconocida como una cantidad con distribución previa, y actualizarla con la verosimilitud de lo observado. En forma de odds, la actualización es una multiplicación por la razón de verosimilitud.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- De Moivre (1718), The Doctrine of Chances
- Jacob Bernoulli (1713), ley de los grandes números


## 4. Intuición

Una prueba acierta el 99 % de las veces. Das positivo. ¿Qué probabilidad tienes de estar enfermo? La respuesta intuitiva —99 %— es incorrecta, y a veces por un factor de diez. Falta un dato que la pregunta no menciona: cuánta gente está enferma.


## 5. Concepto mínimo

```text
P(H|D) = P(D|H)·P(H) / P(D)

En odds, que es donde se ve mejor:
    odds posteriores = odds previas × razón de verosimilitud
    razón de verosimilitud = sensibilidad / (1 − especificidad)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('bayes', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. Con prevalencia 0,001 y prueba del 99 %, ¿qué probabilidad hay tras un positivo?
2. ¿Y con prevalencia 0,40, con la MISMA prueba?
3. ¿Qué pasa tras tres positivos independientes?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('bayes', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('bayes', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con prevalencia 0,001 la respuesta es **0,0902**: de cada 100 positivos, 91 están sanos. Con prevalencia 0,40 y la misma prueba sube a **0,9851**. La prueba no cambió; cambió a quién se le aplica. Y tres positivos independientes llevan de 0,001 a 0,999.


## 10. Comentario pedagógico

Este es el error de razonamiento más caro y más común, y tiene nombre: negligencia de la tasa base. Aparece en cribados médicos, en detección de fraude y en cualquier sistema que busque algo raro. Un modelo con 99 % de exactitud sobre un evento del 0,1 % genera avisos que son falsos nueve de cada diez veces, y eso no es un fallo del modelo.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer P(dato | hipótesis) como si fuese P(hipótesis | dato).


In [ ]:
print('P(positivo | enfermo) = 0,99   <- lo que mide la prueba')
print('P(enfermo | positivo) = 0,09   <- lo que le importa al paciente')
print('Son numeros distintos y confundirlos se llama falacia del fiscal.')

## 12. Corrección

La cuenta completa, con la tasa base delante:


In [ ]:
r = run_paper_lab('bayes', seed=7)['result']
for e in r['escenarios']:
    print(f"{e['caso']:<36} prevalencia={e['prevalencia']:<7} "
          f"P(enfermo|+)={e['p_enfermo_dado_positivo']}")
print()
print('simulacion sobre 100 000 personas:', r['simulacion_sobre_100000_personas'])

## 13. Desafío guiado

Comprueba en la simulación que contar personas y aplicar la fórmula dan lo mismo, y explica por qué contar resulta más fácil de entender.


In [ ]:
r = run_paper_lab('bayes', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica la regla a un detector de fraude de tu trabajo: estima la tasa base real, la sensibilidad y la especificidad, y calcula qué proporción de las alertas serán falsas. Después compara con lo que reporta el equipo.


## 15. Evidencia de aprendizaje

Guarda la tabla de escenarios con su probabilidad posterior y tu explicación de por qué la tasa base cambia la respuesta.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P87_bayes/README.md) · evaluación formal: [`assessments/papers/P87_bayes.md`](../../assessments/papers/P87_bayes.md)


## 16. Cierre

Ya sabemos actualizar una creencia. Falta justificar por qué hay que usar probabilidad y no cualquier otra escala de plausibilidad.


## 17. Conexión con el siguiente hito

- P88
- P91

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
